# Test de bout en bout : Hotel RAG Assistant

À lancer **avant de pousser sur GitHub**. Exécute les cellules dans l'ordre (*Run All*) : chaque cellule vérifie une couche du projet et s'arrête avec une erreur explicite si quelque chose ne va pas. La dernière cellule fait le bilan.

**Prérequis**
1. Noyau (kernel) = le `.venv` du projet, où `pip install -e ".[dev]"` a été exécuté
2. Fichier `.env` avec `OPENAI_API_KEY`
3. PDF dans `data/`

Le test fait quelques appels à l'API OpenAI (embeddings et gpt-4o-mini), pour un coût négligeable. Il **reconstruit l'index Chroma** (`.chroma/`), comme `scripts/ingest.py`.

| # | Étape | Ce qui est vérifié |
|---|---|---|
| 1 | setup | package importable depuis le projet, clé API, PDF présents |
| 2 | ingestion | extraction des rubriques depuis les PDF |
| 3 | openai | embeddings et chat répondent |
| 4 | ingest_script | `scripts/ingest.py` construit l'index |
| 5 | search | Chroma retrouve la bonne rubrique |
| 6 | evaluate_script | `scripts/evaluate.py` tourne, le jeu de test est cohérent |
| 7 | pipeline | réponse RAG complète, refus hors périmètre |
| 8 | threshold | le seuil refuse sans appeler le LLM, et ne bloque pas les questions valides |
| 9 | gradio_respond | la fonction de l'interface gère l'historique et les sources |
| 10 | pytest | tests unitaires |
| 11 | hygiene | aucun secret dans les fichiers, `.env` et `.chroma/` ignorés par git |

In [2]:
import sys, subprocess, importlib.util

print("Noyau          :", sys.executable)
print("Version        :", sys.version.split()[0])
print("find_spec      :", importlib.util.find_spec("hotel_rag"))

fresh = subprocess.run(
    [sys.executable, "-I", "-c", "import hotel_rag; print(hotel_rag.__file__)"],
    capture_output=True, text=True,
)
print("Processus neuf :", fresh.stdout.strip() or fresh.stderr.strip().splitlines()[-1])

Noyau          : /home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/.venv/bin/python
Version        : 3.13.3
find_spec      : ModuleSpec(name='hotel_rag', loader=<_frozen_importlib_external.SourceFileLoader object at 0x74c505eaa0f0>, origin='/home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/src/hotel_rag/__init__.py', submodule_search_locations=['/home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/src/hotel_rag'])
Processus neuf : /home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/src/hotel_rag/__init__.py


In [3]:
import sys, site
print("\n".join(sys.path))
print(site.getsitepackages())

/home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel
/home/isoilahoudin/.local/share/uv/python/cpython-3.13.3-linux-x86_64-gnu/lib/python313.zip
/home/isoilahoudin/.local/share/uv/python/cpython-3.13.3-linux-x86_64-gnu/lib/python3.13
/home/isoilahoudin/.local/share/uv/python/cpython-3.13.3-linux-x86_64-gnu/lib/python3.13/lib-dynload

/home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/.venv/lib/python3.13/site-packages
/home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/src
['/home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/.venv/lib/python3.13/site-packages']


In [4]:
import os
import json
import re
import subprocess
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # pour pouvoir faire `import app`

report = {}      # étapes validées
warnings = []    # points à regarder, non bloquants

try:
    import hotel_rag
except ImportError as exc:
    raise RuntimeError(
        "hotel_rag introuvable : sélectionne le noyau .venv du projet (en haut à droite) "
        'et vérifie que `pip install -e ".[dev]"` a été lancé dans ce venv.'
    ) from exc

from hotel_rag import config

pkg_dir = Path(hotel_rag.__file__).resolve().parent
assert ROOT.resolve() in pkg_dir.parents, f"hotel_rag est importé depuis {pkg_dir}, pas depuis ce projet"

key = os.getenv("OPENAI_API_KEY", "")
assert key, "OPENAI_API_KEY absente : crée .env à partir de .env.example"
assert key.startswith("sk-"), "OPENAI_API_KEY ne ressemble pas à une clé OpenAI"

pdfs = sorted(config.DATA_DIR.glob("*.pdf"))
assert pdfs, f"Aucun PDF dans {config.DATA_DIR}"

if config.SIMILARITY_THRESHOLD == 0.0:
    warnings.append("SIMILARITY_THRESHOLD = 0.0 : seuil désactivé (calibre-le avant de publier)")

print("Racine :", ROOT)
print("Package:", pkg_dir)
print("Modèles:", config.LLM_MODEL, "/", config.EMBEDDING_MODEL)
print("Seuil  :", config.SIMILARITY_THRESHOLD, "(0.0 = désactivé)")
print("PDF    :", len(pdfs), "fichier(s)")
report["setup"] = True

Racine : /home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel
Package: /home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/src/hotel_rag
Modèles: gpt-4o-mini / text-embedding-3-small
Seuil  : 0.0 (0.0 = désactivé)
PDF    : 5 fichier(s)


## 2. Ingestion des PDF

In [5]:
from hotel_rag.ingestion import load_sections

sections = load_sections(config.DATA_DIR)
print(f"{len(sections)} rubriques extraites de {len(pdfs)} PDF\n")
for s in sections:
    print(f"- [{s.source}] {s.title}  ({len(s.text.split())} mots)")

assert sections, "Aucune rubrique extraite"
too_short = [s.title for s in sections if len(s.text.split()) < 5]
assert not too_short, f"Rubriques quasi vides (extraction ratée ?) : {too_short}"

titles = [s.title for s in sections]
if len(set(titles)) != len(titles):
    warnings.append("Titres de rubriques en double : evaluate.py classe les rubriques par titre")
report["ingestion"] = True

15 rubriques extraites de 5 PDF

- [activites_et_evenements.pdf] Activités et alentours  (149 mots)
- [activites_et_evenements.pdf] Séminaires et événements  (76 mots)
- [chambres_et_tarifs.pdf] Chambres et tarifs  (214 mots)
- [chambres_et_tarifs.pdf] Politique d'annulation et de paiement  (111 mots)
- [chambres_et_tarifs.pdf] Climatisation et confort des chambres  (62 mots)
- [familles_animaux_accessibilite.pdf] Familles et enfants  (78 mots)
- [familles_animaux_accessibilite.pdf] Animaux domestiques  (72 mots)
- [familles_animaux_accessibilite.pdf] Accessibilité PMR  (67 mots)
- [informations_pratiques.pdf] Horaires de la réception  (91 mots)
- [informations_pratiques.pdf] Wifi et connexion internet  (67 mots)
- [informations_pratiques.pdf] Parking et navette  (74 mots)
- [informations_pratiques.pdf] Blanchisserie et services annexes  (59 mots)
- [restaurant_et_bien_etre.pdf] Petit-déjeuner et restaurant  (78 mots)
- [restaurant_et_bien_etre.pdf] Piscine  (64 mots)
- [restaurant_et_

## 3. Connexion à OpenAI

In [3]:
from hotel_rag.llm import chat, embed

vectors = embed(["Bonjour", "Hello"])
assert len(vectors) == 2, "embed doit renvoyer un vecteur par texte"
dim = len(vectors[0])
norm = sum(x * x for x in vectors[0]) ** 0.5
print(f"Embeddings : {len(vectors)} vecteurs de dimension {dim}, norme {norm:.4f}")

if config.EMBEDDING_MODEL == "text-embedding-3-small":
    assert dim == 1536, f"Dimension inattendue : {dim}"
if abs(norm - 1) > 1e-3:
    warnings.append(f"Vecteurs non normalisés (norme {norm:.3f}) : vérifie le modèle d'embedding")

reply = chat([{"role": "user", "content": "Réponds uniquement par le mot OK."}], max_tokens=5)
print("Chat :", repr(reply))
assert reply and reply.strip(), "Réponse vide du LLM"
report["openai"] = True

Embeddings : 2 vecteurs de dimension 1536, norme 1.0001
Chat : 'OK'


## 4. Script d'ingestion (`scripts/ingest.py`)

Lancé dans un sous-processus, comme en ligne de commande. Le client Chroma de ce noyau n'est pas encore ouvert à ce stade.

In [4]:
proc = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "ingest.py")],
    cwd=ROOT, capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-2000:])
assert proc.returncode == 0, "scripts/ingest.py a échoué (voir l'erreur ci-dessus)"
assert f"{len(sections)} rubriques indexées" in proc.stdout, "Le nombre de rubriques indexées ne correspond pas"
assert config.CHROMA_DIR.exists(), ".chroma/ n'a pas été créé"

from hotel_rag import store
store._client.cache_clear()   # repart d'un client frais sur l'index qui vient d'être construit
report["ingest_script"] = True

15 rubriques indexées dans /home/isoilahoudin/projects/RAG/projet_rag_assistant_hotel/.chroma



## 5. Recherche vectorielle

In [5]:
from hotel_rag.store import search

cases = [
    ("A quelle heure ouvre la réception ?", "Horaires de la réception"),
    ("Y a-t-il un spa ?", "Spa et bien-être"),
]

known = {s.title for s in sections}
for question, expected in cases:
    assert expected in known, f"Titre attendu absent des PDF : {expected!r} (adapte `cases`)"

    hits = search(question, top_k=config.TOP_K)
    print(question)
    for h in hits:
        print(f"   {h.similarity:.3f}  {h.title}")

    assert len(hits) == config.TOP_K, f"{len(hits)} résultats au lieu de {config.TOP_K}"
    sims = [h.similarity for h in hits]
    assert sims == sorted(sims, reverse=True), "Résultats non triés par similarité décroissante"
    assert hits[0].title == expected, f"Meilleur résultat : {hits[0].title!r}, attendu {expected!r}"

report["search"] = True

A quelle heure ouvre la réception ?
   0.659  Horaires de la réception
   0.446  Petit-déjeuner et restaurant
Y a-t-il un spa ?
   0.634  Spa et bien-être
   0.445  Piscine


## 6. Script d'évaluation (`scripts/evaluate.py`)

In [17]:
eval_path = ROOT / "eval" / "eval_set.json"
assert eval_path.exists(), "eval/eval_set.json introuvable"

items = json.loads(eval_path.read_text(encoding="utf-8"))
from hotel_rag.ingestion import load_sections
known = {s.title for s in load_sections(config.DATA_DIR)}

unknown = {it["expected"] for it in items} - known
assert not unknown, f"Titres attendus inconnus (reste-t-il des '...' ?) : {unknown}"
if len(items) < 20:
    warnings.append(f"Jeu d'évaluation de {len(items)} questions : trop petit pour des métriques stables (vise 20 à 30)")

proc = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "evaluate.py")],
    cwd=ROOT, capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-2000:])
assert proc.returncode == 0, "scripts/evaluate.py a échoué (voir l'erreur ci-dessus)"
report["evaluate_script"] = True

## 7. Pipeline RAG complet

In [7]:
from hotel_rag.generation import FALLBACK
from hotel_rag.pipeline import chat_turn

# Question dont la réponse est dans la documentation (seuil désactivé pour isoler le pipeline)
q = "A quelle heure ouvre la réception ?"
ans = chat_turn(q, [], threshold=0.0)
print("Client   :", q)
print("Rubriques:", [h.title for h in ans.hits])
print("Assistant:", ans.text, "\n")

assert not ans.refused and ans.hits
assert ans.text.strip(), "Réponse vide"
assert ans.text.strip() != FALLBACK, "Le LLM a refusé de répondre alors que l'information est dans la documentation"

# Question hors périmètre : le LLM doit refuser (contrôle souple, la sortie d'un LLM peut varier)
q_out = "Quelle est la capitale de l'Australie ?"
out = chat_turn(q_out, [], threshold=0.0)
print("Client   :", q_out)
print("Assistant:", out.text)
if "Je ne sais pas" not in out.text:
    warnings.append(f"Hors périmètre non refusé par le LLM : {out.text!r}")
report["pipeline"] = True

Client   : A quelle heure ouvre la réception ?
Rubriques: ['Horaires de la réception', 'Petit-déjeuner et restaurant']
Assistant: La réception de l'Hôtel Le Belvédère ouvre à 7h. 

Client   : Quelle est la capitale de l'Australie ?
Assistant: Je ne sais pas, je vous invite à contacter la réception.


In [8]:
# Question de suivi : la réécriture doit rendre la question autonome
history = []
for q in ["Y a-t-il une piscine ?", "Et jusqu'à quelle heure ?"]:
    ans = chat_turn(q, history, threshold=0.0)
    print(f"Client    : {q}")
    print(f"Recherche : {ans.standalone_question}")
    print(f"Rubriques : {[h.title for h in ans.hits]}")
    print(f"Assistant : {ans.text}\n")
    history += [{"role": "user", "content": q}, {"role": "assistant", "content": ans.text}]

if "piscine" not in ans.standalone_question.lower():
    warnings.append(f"Réécriture de la question de suivi sans 'piscine' : {ans.standalone_question!r}")

Client    : Y a-t-il une piscine ?
Recherche : Y a-t-il une piscine ?
Rubriques : ['Piscine', 'Familles et enfants']
Assistant : Oui, l'hôtel dispose d'une piscine extérieure chauffée et d'une piscine intérieure accessible toute l'année.

Client    : Et jusqu'à quelle heure ?
Recherche : Jusqu'à quelle heure est ouverte la piscine ?
Rubriques : ['Piscine', 'Familles et enfants']
Assistant : La piscine extérieure est ouverte de 8h à 20h, tandis que la piscine intérieure est accessible de 7h à 21h.



## 8. Seuil de similarité

In [9]:
from unittest.mock import patch

q = "A quelle heure ouvre la réception ?"

# 1. Seuil impossible à atteindre : refus, et le LLM ne doit PAS être appelé
with patch("hotel_rag.pipeline.chat", side_effect=AssertionError("LLM appelé alors que le seuil n'est pas atteint")):
    refused = chat_turn(q, [], threshold=1.01)
assert refused.refused and refused.text == FALLBACK
print("Seuil 1.01 -> refus sans appel LLM :", refused.text)

# 2. Valeur configurée : ne doit pas bloquer une question valide
ok = chat_turn(q, [])
best = ok.hits[0].similarity
print(f"Seuil configuré = {config.SIMILARITY_THRESHOLD} ; similarité de la question valide = {best:.3f}")
assert not ok.refused, f"Le seuil configuré ({config.SIMILARITY_THRESHOLD}) bloque une question valide"
if config.SIMILARITY_THRESHOLD and best - config.SIMILARITY_THRESHOLD < 0.05:
    warnings.append(f"Seuil très proche d'une question valide (marge {best - config.SIMILARITY_THRESHOLD:.3f})")
report["threshold"] = True

Seuil 1.01 -> refus sans appel LLM : Je ne sais pas, je vous invite à contacter la réception.
Seuil configuré = 0.0 ; similarité de la question valide = 0.659


## 9. Fonction de l'interface Gradio

On teste `respond` directement : l'interface n'est pas lancée (voir le test manuel en bas).

In [12]:
from hotel_rag.app import demo, respond, SEP

print("Interface :", type(demo).__name__)

out = respond("A quelle heure ouvre la réception ?", [])
print(out, "\n")
assert isinstance(out, str) and "Sources" in out, "Réponse sans sources"

# Historique au format Gradio, avec la ligne de sources que respond doit retirer avant de l'envoyer au LLM
history = [
    {"role": "user", "content": "Y a-t-il une piscine ?"},
    {"role": "assistant", "content": "Oui." + SEP + "Piscine*"},
]
out2 = respond("Et jusqu'à quelle heure ?", history)
print(out2)
assert isinstance(out2, str) and out2.strip()
report["gradio_respond"] = True

Interface : ChatInterface
La réception de l'Hôtel Le Belvédère ouvre à 7h.

---
*Sources : Horaires de la réception, Petit-déjeuner et restaurant* 

La piscine extérieure est ouverte de 8h à 20h et la piscine intérieure de 7h à 21h.

---
*Sources : Piscine, Familles et enfants*


## 10. Tests unitaires

In [13]:
proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=ROOT, capture_output=True, text=True,
)
print(proc.stdout[-2500:])
if proc.returncode != 0:
    print(proc.stderr[-1500:])
assert proc.returncode == 0, "pytest a échoué (si 'No module named pytest' : pip install -e \".[dev]\")"
report["pytest"] = True

......                                                                   [100%]
6 passed in 1.22s



## 11. Hygiène du dépôt (avant `git push`)

In [14]:
SKIP_DIRS = {".git", ".venv", "venv", ".chroma", "__pycache__", ".pytest_cache", "node_modules"}
SKIP_SUFFIXES = {".pdf", ".sqlite3", ".bin", ".png", ".jpg", ".jpeg", ".gif"}
pattern = re.compile(r"sk-[A-Za-z0-9_\-]{20,}")

leaks = []
for path in ROOT.rglob("*"):
    rel = path.relative_to(ROOT)
    if any(part in SKIP_DIRS for part in rel.parts) or not path.is_file():
        continue
    if rel.name == ".env" or path.suffix in SKIP_SUFFIXES:
        continue
    try:
        text = path.read_text(encoding="utf-8")
    except (UnicodeDecodeError, OSError):
        continue
    if pattern.search(text):
        leaks.append(str(rel))
assert not leaks, f"Clé API probable dans : {leaks} (supprime-la, puis révoque la clé)"
print("Aucune clé trouvée dans les fichiers du projet")

def git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)

if git("rev-parse", "--is-inside-work-tree").returncode != 0:
    warnings.append("Pas de dépôt git ici : vérifications .gitignore sautées (git init)")
else:
    for name in (".env", ".chroma"):
        rc = git("check-ignore", "-q", name).returncode
        assert rc == 0, f"{name} n'est pas ignoré par git : ajoute-le à .gitignore"
    tracked = git("ls-files", ".env", ".chroma").stdout.strip()
    assert not tracked, f"Fichiers déjà suivis par git, à retirer avec `git rm --cached` : {tracked}"
    print(".env et .chroma/ sont ignorés et non suivis par git")
report["hygiene"] = True

Aucune clé trouvée dans les fichiers du projet
.env et .chroma/ sont ignorés et non suivis par git


## Bilan

In [15]:
expected = {
    "setup": "Environnement, clé API, PDF",
    "ingestion": "Extraction des rubriques",
    "openai": "Embeddings et chat OpenAI",
    "ingest_script": "scripts/ingest.py",
    "search": "Recherche Chroma",
    "evaluate_script": "scripts/evaluate.py et jeu de test",
    "pipeline": "Pipeline RAG complet",
    "threshold": "Seuil de similarité",
    "gradio_respond": "Fonction de l'interface Gradio",
    "pytest": "Tests unitaires",
    "hygiene": "Secrets et .gitignore",
}

for k, label in expected.items():
    print(f"{'✅' if k in report else '❌'} {label}")

if warnings:
    print("\nÀ regarder (non bloquant) :")
    for w in warnings:
        print(f"  ⚠️  {w}")

if all(k in report for k in expected):
    print("\nTout est vert : tu peux pousser.")
else:
    print("\nCertaines étapes ne sont pas validées : corrige la première cellule en erreur, puis relance.")

✅ Environnement, clé API, PDF
✅ Extraction des rubriques
❌ Embeddings et chat OpenAI
❌ scripts/ingest.py
❌ Recherche Chroma
❌ scripts/evaluate.py et jeu de test
✅ Pipeline RAG complet
✅ Seuil de similarité
✅ Fonction de l'interface Gradio
✅ Tests unitaires
✅ Secrets et .gitignore

À regarder (non bloquant) :
  ⚠️  SIMILARITY_THRESHOLD = 0.0 : seuil désactivé (calibre-le avant de publier)

Certaines étapes ne sont pas validées : corrige la première cellule en erreur, puis relance.


## Test manuel de l'interface (2 minutes)

Les cellules ci-dessus ne lancent pas l'interface. Dans un terminal, `python app.py`, puis vérifie dans le navigateur (`http://127.0.0.1:7860`) :

1. Une question valide donne une réponse **avec** la ligne « Sources »
2. Une question de suivi (« Et jusqu'à quelle heure ? ») est comprise
3. Une question hors périmètre donne « Je ne sais pas… » **sans** sources
4. Les exemples cliquables fonctionnent